# Prompt Ablation - contraTICO
## 1) Source QA

Generate source answers with P1-fewshot, P2-cot, P3-concise for all 3 configs.

**Runs:** 3 strategies × 3 configs = 9 runs

In [ ]:
import os
import sys
import subprocess

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = os.path.exists('/kaggle')

if IN_KAGGLE:
    PROJECT_ROOT = '/kaggle/working/askqe'
    if not os.path.exists(PROJECT_ROOT):
        subprocess.run(['git', 'clone',
                        'https://github.com/Simone280802/AskQE_DNLP_2025-2026.git',
                        PROJECT_ROOT], check=True)
elif IN_COLAB:
    PROJECT_ROOT = '/content/askqe'
    if not os.path.exists(PROJECT_ROOT):
        subprocess.run(['git', 'clone',
                        'https://github.com/Simone280802/AskQE_DNLP_2025-2026.git',
                        PROJECT_ROOT], check=True)
else:
    PROJECT_ROOT = os.getcwd()

print(f'Project root: {PROJECT_ROOT}')

# ─── Paths ───
BASELINE_DIR = f"{PROJECT_ROOT}/results Qwen3B baseline/contratico/baseline"
OUTPUT_DIR = f"{PROJECT_ROOT}/results Qwen3B baseline/contratico/prompt-ablation"
if IN_KAGGLE or IN_COLAB:
     OUTPUT_DIR = "/kaggle/working/prompt-ablation" if IN_KAGGLE else "/content/prompt-ablation"
     
CODE_DIR = f"{PROJECT_ROOT}/results Qwen3B baseline/contratico/prompt-ablation/code"

STRATEGIES = ["P1-fewshot", "P2-cot", "P3-concise"]
CONFIGS = ["vanilla", "atomic", "semantic"]

print(f"Baseline: {BASELINE_DIR}")
print(f"Output: {OUTPUT_DIR}")
print(f"Code: {CODE_DIR}")

In [ ]:
# Run all source QA (3 strategies × 3 configs = 9 runs)
for strategy in STRATEGIES:
    for config in CONFIGS:
        print(f"\n{'=' * 60}")
        print(f"Source QA: {strategy} / {config}")
        print(f"{'=' * 60}")
        
        result = subprocess.run(
            [
                sys.executable, f"{CODE_DIR}/qa_ablation_contratico.py",
                "--strategy", strategy,
                "--mode", "source",
                "--config", config,
                "--baseline_dir", BASELINE_DIR,
                "--output_dir", OUTPUT_DIR,
                "--max_rows", "42",
                "--seed", "42",
            ],
            capture_output=True, text=True
        )
        print(result.stdout)
        if result.returncode != 0:
            print(f"ERROR: {result.stderr}")

In [ ]:
# Verify output files
for strategy in STRATEGIES:
    for config in CONFIGS:
        path = f"{OUTPUT_DIR}/QA/{strategy}/source/en-{config}.jsonl"
        if os.path.exists(path):
            count = sum(1 for line in open(path))
            print(f"✓ {strategy}/source/en-{config}.jsonl: {count} rows")
        else:
            print(f"✗ MISSING: {path}")